In [1]:
import copy
import datetime
import numpy as np
import os
import pandas as pd
import polars as pl
import requests
import zipfile

from tqdm import tqdm
from typing import Tuple, Dict, List, Any

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
def download(url: str, filename: str, chunk_size: int = 1024):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)

    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall(".")

In [3]:
download(
    url="http://files.grouplens.org/datasets/movielens/ml-1m.zip",
    filename="ml-1m.zip"
)

In [4]:
dataset_path = './ml-1m/'

ratings = pd.read_csv(
    os.path.join(dataset_path, 'ratings.dat'),
    delimiter='::',
    header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    engine='python'
)
ratings = pl.from_pandas(ratings)

movies = pd.read_csv(
    os.path.join(dataset_path, 'movies.dat'),
    delimiter='::',
    header=None,
    names=['movie_id', 'title', 'genres'],
    encoding='latin-1',
    engine='python'
)
movies = pl.from_pandas(movies)

users = pd.read_csv(
    os.path.join(dataset_path, 'users.dat'),
    delimiter='::',
    header=None,
    names=['user_id', 'gender', 'age', 'occupation', 'zip_code'],
    engine='python'
)
users = pl.from_pandas(users)

In [5]:
filtering_stage = 0
is_changed = True
threshold = 5
good_users = set()
good_items = set()

filtered_ratings = ratings.clone()

# фильтруем данные
while is_changed:
    user_counts = filtered_ratings.group_by('user_id').agg(pl.len().alias('user_count'))
    movie_counts = filtered_ratings.group_by('movie_id').agg(pl.len().alias('movie_count'))

    good_users = user_counts.filter(pl.col('user_count') >= threshold).select('user_id')
    good_movies = movie_counts.filter(pl.col('movie_count') >= threshold).select('movie_id')

    old_size = len(filtered_ratings)

    new_ratings = filtered_ratings.join(good_users, on='user_id', how='inner')
    new_ratings = new_ratings.join(good_movies, on='movie_id', how='inner')

    new_size = len(new_ratings)

    filtered_ratings = new_ratings
    is_changed = old_size != new_size
    filtering_stage += 1

filtered_ratings = filtered_ratings.with_columns(movie_id = pl.col('movie_id').rank('dense') - 1)
filtered_ratings = filtered_ratings.sort(['user_id', 'timestamp'])

grouped_filtered_df = filtered_ratings.group_by('user_id', maintain_order=True).agg(
    pl.all().exclude('user_id')
)

In [6]:
valid_portion = 0.1
test_portion = 0.1

all_events_timestamp = []
for _, _, _, timestamp in filtered_ratings.iter_rows():
    all_events_timestamp.append(timestamp)

all_events_timestamp = sorted(all_events_timestamp)

fst_threshold = all_events_timestamp[int(len(all_events_timestamp) * (1.0 - test_portion - valid_portion))]
snd_threshold = all_events_timestamp[int(len(all_events_timestamp) * (1.0 - test_portion))]


In [7]:
train_data = []
valid_data = []
eval_data = []


for user_id, item_history, rating, timestamp in grouped_filtered_df.iter_rows():
    train_history = []
    history = []
    history_ts = []

    for item_id, ts in zip(item_history, timestamp):
        # этап обучения
        if ts < fst_threshold:
            assert len(history) == 0 or ts >= history_ts[-1]
            train_history.append(item_id)
        # этап валидации
        elif ts < snd_threshold:
            assert len(history) == 0 or ts >= history_ts[-1]
            if len(history) >= 1:
                valid_data.append({
                    'user_id': user_id,
                    'history': [x for x in history + [item_id]]
                })
        else:
            assert len(history) == 0 or ts >= history_ts[-1]
            if len(history) >= 1:
                eval_data.append({
                    'user_id': user_id,
                    'history': [x for x in history + [item_id]]
                })

        history.append(item_id)
        history_ts.append(ts)

    if len(train_history) >= 5:
        train_data.append({
            'user_id': user_id,
            'history': train_history
        })

In [8]:
class MovieLensDataset(Dataset):
    def __init__(
            self,
            data: List[Dict],
            num_items: int,
            max_seq_len: int,
        ) -> None:
        super().__init__()
        self.data = data
        self.num_items = num_items
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Dict[str, List[int]]:
        sample = self.data[index]
        items = list(map(int, sample['history']))

        result = {}

        item_sequence = items[:-1][-self.max_seq_len:]
        next_item_sequence = items[1:][-self.max_seq_len:]

        random_negative_ids = np.random.randint(0, self.num_items, size=(len(item_sequence,))).tolist()

        result['history'] = dict(
            item_ids=item_sequence,
            lengths=len(item_sequence),
            positions=np.arange(start=len(item_sequence) - 1, stop=-1, step=-1).tolist()
        )

        # выделяем позитивные и нагативные элементы
        result['positives'] = dict(
            item_ids=next_item_sequence
        )
        result['negatives'] = dict(
            item_ids=random_negative_ids
        )

        return result

In [9]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    result = {}
    keys = batch[0].keys()
    for key in keys:
        values = [item[key] for item in batch]
        first = values[0]
        if isinstance(first, dict):
            result[key] = collate_fn(values)
        elif isinstance(first, list):
            result[key] = torch.tensor(sum(values, []), dtype=torch.long)
        else:
            result[key] = torch.tensor(values, dtype=torch.long)
    return result

In [10]:
NUM_ITEMS = 3416
MAX_SEQ_LEN = 5

tmp_dataset = MovieLensDataset(data=train_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)

In [11]:
tmp_iter = iter(tmp_dataset)

In [12]:
BATCH_SIZE = 3

tmp_dataloader = DataLoader(dataset=tmp_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [13]:
def get_mask(lengths: torch.Tensor) -> torch.Tensor:
    maxlen = lengths.max().item()
    return torch.arange(maxlen, device=lengths.device).expand(len(lengths), maxlen) < lengths.unsqueeze(1)


def get_last(data: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    offsets = torch.cumsum(lengths, dim=-1)
    return data[offsets - 1]

In [14]:
class SasRecBackbone(nn.Module):
    def __init__(
            self,
            num_items: int,
            embedding_dim: int = 64,
            num_heads: int = 2,
            max_seq_len: int = 512,
            dropout_rate: float = 0.2,
            num_transformer_layers: int = 2,
        ) -> None:
        super().__init__()
        self.num_items = num_items
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.item_encoder = nn.Embedding(
            num_embeddings=num_items,
            embedding_dim=embedding_dim
        )
        self.position_embeddings = nn.Embedding(
            num_embeddings=max_seq_len,
            embedding_dim=embedding_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=dropout_rate,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_transformer_layers)

    @property
    def catalog_embeddings(self) -> torch.Tensor:
        return self.item_encoder.weight.data

    def forward(self, inputs: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        item_embeddings = self.item_encoder(inputs['history']['item_ids'])  # (total_batch_events, embedding_dim)
        padding_mask = get_mask(inputs['history']['lengths'])  # (batch_size, seq_len)
        batch_size, seq_len = padding_mask.shape

        token_embeddings = item_embeddings.new_zeros(
            batch_size, seq_len, self.embedding_dim
        )  # (batch_size, seq_len, embedding_dim)
        token_embeddings[padding_mask] = (
            item_embeddings
            + self.position_embeddings(inputs['history']['positions'])
        )
        encoder_output = self.transformer_encoder(
            src=token_embeddings,
            mask=torch.triu(
                torch.ones((seq_len, seq_len), dtype=torch.bool, device=token_embeddings.device),
                diagonal=1
            ),
            src_key_padding_mask=~padding_mask,
            is_causal=True
        )[padding_mask]

        return {
            'encoder_output': encoder_output,
        }

In [15]:
class SASRecModel(nn.Module):
    def __init__(
            self,
            backbone: SasRecBackbone,
        ) -> None:
        super().__init__()
        self.backbone = backbone
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float = 0.02) -> None:
        for key, value in self.named_parameters():
            if 'weight' in key:
                if 'norm' in key:
                    nn.init.ones_(value.data)
                else:
                    nn.init.trunc_normal_(
                        value.data,
                        std=initializer_range,
                        a=-2 * initializer_range,
                        b=2 * initializer_range
                    )
            elif 'bias' in key:
                nn.init.zeros_(value.data)
            else:
                raise ValueError(f'Unknown weight: {key}')

    def compute_loss(self, inputs: Dict, backbone_output: Dict) -> Dict:
        raise NotImplementedError

    def forward(self, inputs: Dict):
        backbone_outputs = self.backbone(inputs)

        if self.training:
            loss = self.compute_loss(inputs, backbone_outputs)
            return {
                'loss': loss
            }
        else:
            last_embeddings = get_last(
                data=backbone_outputs['encoder_output'],
                lengths=inputs['history']['lengths']
            )
            candidate_scores = last_embeddings @ self.backbone.catalog_embeddings.T
            return {
                'candidate_scores': candidate_scores
            }

In [16]:
class SasRecReal(SASRecModel):
    def compute_loss(self, inputs: Dict, backbone_output: Dict) -> torch.Tensor:
        query_embeddings = backbone_output['encoder_output']
        positive_embeddings = self.backbone.item_encoder(inputs['positives']['item_ids'])
        negative_embeddings = self.backbone.item_encoder(inputs['negatives']['item_ids'])

        positive_scores = (query_embeddings * positive_embeddings).sum(dim=-1)
        negative_scores = (query_embeddings * negative_embeddings).sum(dim=-1)

        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            input=torch.cat([positive_scores, negative_scores], dim=0),
            target=torch.cat([torch.ones_like(positive_scores), torch.zeros_like(negative_scores)]),
            reduction='mean'
        )

        return loss

In [18]:
# функции подсчета метрик
def compute_hitrate(indices: torch.Tensor, labels: torch.Tensor, k: int) -> List[float]:
    assert indices.shape[0] == labels.shape[0]
    hits = torch.eq(indices[:, :k], labels[..., None]).float()
    hitrate = hits.sum(dim=-1)
    return np.mean(hitrate.cpu().tolist())


def compute_dcg(indices: torch.Tensor, labels: torch.Tensor, k: int) -> List[float]:
    assert indices.shape[0] == labels.shape[0]
    hits = torch.eq(indices[:, :k], labels[..., None]).float()
    discount_factor = 1 / torch.log2(torch.arange(1, k + 1, 1).float() + 1.).to(hits.device)
    dcg = torch.einsum('bk,k->b', hits, discount_factor)
    return np.mean(dcg.cpu().tolist())


def compute_coverage(indices: torch.Tensor, k: int, num_items: int = NUM_ITEMS) -> List[float]:
    return indices[:, :k].reshape(-1).unique().shape[0] / num_items


def compute_metrics(indices: torch.Tensor, labels: torch.Tensor) -> Dict[str, float]:
    return {
        'dcg@5': compute_dcg(indices, labels, k=5),
        'dcg@10': compute_dcg(indices, labels, k=10),
        'dcg@20': compute_dcg(indices, labels, k=20),

        'hitrate@5': compute_hitrate(indices, labels, k=5),
        'hitrate@10': compute_hitrate(indices, labels, k=10),
        'hitrate@20': compute_hitrate(indices, labels, k=20),

        'coverage@5': compute_coverage(indices, k=5),
        'coverage@10': compute_coverage(indices, k=10),
        'coverage@20': compute_coverage(indices, k=20),
    }

In [19]:
def evaluation(
        dataloader: DataLoader,
        model: SASRecModel,
        device: str = 'cpu',
    ) -> Tuple[torch.Tensor, torch.Tensor]:
    outputs = []
    labels = []

    for batch in dataloader:
        model.eval()

        for key in batch:
            if isinstance(batch[key], dict):
                for sub_key in batch[key]:
                    batch[key][sub_key] = batch[key][sub_key].to(device)
            else:
                assert isinstance(batch[key], torch.Tensor)
                batch[key] = batch[key].to(device)

        candidate_scores = model(batch)['candidate_scores']
        _, top_candiate_ids = torch.topk(
            candidate_scores,
            k=20, dim=-1, largest=True
        )
        next_item_ids = get_last(
            data=batch['positives']['item_ids'],
            lengths=batch['history']['lengths']
        )
        outputs.append(top_candiate_ids)
        labels.append(next_item_ids)

    return torch.cat(outputs, dim=0), torch.cat(labels, dim=0)

In [20]:
def train(
        train_dataloader: DataLoader,
        valid_dataloader: DataLoader,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int | None = None,
        device: str = 'cpu'
    ) -> torch.nn.Module:
    step_num = 0
    epoch_num = 0

    best_checkpoint = None
    best_metric_name = 'dcg@20'
    best_metric_value = float('-inf')

    while num_epochs is None or epoch_num < num_epochs:
        running_loss = []
        for batch in train_dataloader:
            model.train()

            for key in batch:
                if isinstance(batch[key], dict):
                    for sub_key in batch[key]:
                        batch[key][sub_key] = batch[key][sub_key].to(device)
                else:
                    assert isinstance(batch[key], torch.Tensor)
                    batch[key] = batch[key].to(device)

            loss = model(batch)['loss']

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            step_num += 1

            running_loss.append(loss.item())

            # Запускаем валидацию
            if step_num % 64 == 0:
                pred_valid_indices, true_valid_labels = evaluation(valid_dataloader, model, device)
                valid_metrics = compute_metrics(pred_valid_indices, true_valid_labels)

                if best_metric_value is None or best_metric_value < valid_metrics[best_metric_name]:
                    best_metric_value = valid_metrics[best_metric_name]
                    best_checkpoint = copy.deepcopy(model.state_dict())

                msgs = []
                for metric_name, metrinc_value in valid_metrics.items():
                    msgs.append(f'{metric_name}: {round(metrinc_value, 5)}')
                msg = ', '.join(msgs)
                print(msg)

        print(f'Средний лосс на эпохе #{epoch_num + 1}: {round(np.mean(running_loss), 5)}')

        epoch_num += 1

    print('Обучение завершено!')

    return best_checkpoint

In [21]:
LEARNING_RATE = 0.001
BATCH_SIZE = 256
NUM_EPOCHS = 10

EMBEDDING_DIM = 64
NUM_HEADS = 2
MAX_SEQ_LEN = 200
DROPOUT_RATE = 0.1
NUM_TRANSFORMER_LAYERS = 2

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [22]:
backbone = SasRecBackbone(
    num_items=NUM_ITEMS,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    max_seq_len=MAX_SEQ_LEN,
    num_transformer_layers=NUM_TRANSFORMER_LAYERS,
)

In [23]:
train_dataset = MovieLensDataset(data=train_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)
valid_dataset = MovieLensDataset(data=valid_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)
eval_dataset = MovieLensDataset(data=eval_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=collate_fn
)
valid_dataloader = DataLoader(
    dataset=valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn
)
eval_dataloader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn
)

model = SasRecReal(
    backbone=backbone,
).to(DEVICE)

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

best_checkpoint = train(
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    model=model,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=DEVICE
)
model.load_state_dict(best_checkpoint)

pred_eval_indices, true_eval_labels = evaluation(eval_dataloader, model, device=DEVICE)
eval_metrics = compute_metrics(pred_eval_indices, true_eval_labels)

msgs = []
for metric_name, metrinc_value in eval_metrics.items():
    msgs.append(f'{metric_name}: {round(metrinc_value, 5)}')
msg = ', '.join(msgs)
print(msg)

checkpoint_path = './sasrec_full.pth'
torch.save(best_checkpoint, checkpoint_path)

Средний лосс на эпохе #1: 0.64035
Средний лосс на эпохе #2: 0.54346
Средний лосс на эпохе #3: 0.52794
dcg@5: 0.00924, dcg@10: 0.01402, dcg@20: 0.02016, hitrate@5: 0.01509, hitrate@10: 0.03009, hitrate@20: 0.05459, coverage@5: 0.11036, coverage@10: 0.16393, coverage@20: 0.2377
Средний лосс на эпохе #4: 0.51754
Средний лосс на эпохе #5: 0.49473
Средний лосс на эпохе #6: 0.45488
dcg@5: 0.02352, dcg@10: 0.03323, dcg@20: 0.04508, hitrate@5: 0.03898, hitrate@10: 0.06924, hitrate@20: 0.11652, coverage@5: 0.20872, coverage@10: 0.26347, coverage@20: 0.3308
Средний лосс на эпохе #7: 0.42261
Средний лосс на эпохе #8: 0.40218
Средний лосс на эпохе #9: 0.38619
dcg@5: 0.03167, dcg@10: 0.04564, dcg@20: 0.06241, hitrate@5: 0.05376, hitrate@10: 0.09741, hitrate@20: 0.16424, coverage@5: 0.33402, coverage@10: 0.39432, coverage@20: 0.46341
Средний лосс на эпохе #10: 0.37301
Обучение завершено!
dcg@5: 0.02185, dcg@10: 0.0316, dcg@20: 0.04397, hitrate@5: 0.03776, hitrate@10: 0.06822, hitrate@20: 0.11749, co